In [ ]:
import zipfile
import sqlite3
import pandas as pd
import json
import os

# Unzip the .apkg file (it's a zip archive)
apkg_path = 'Deutsch_4000_German_Words_by_Frequency.apkg'
extract_dir = 'apkg_extracted'

# with zipfile.ZipFile(apkg_path, 'r') as zip_ref:
#     zip_ref.extractall(extract_dir)

# print('Extracted files:', os.listdir(extract_dir))

: 

In [ ]:
# Connect to the SQLite database
db_path = os.path.join(extract_dir, 'collection.anki2')
conn = sqlite3.connect(db_path)

# List all tables
tables = pd.read_sql("SELECT name FROM sqlite_master WHERE type='table';", conn)
print('Tables in database:')
print(tables)

In [ ]:
# Load the notes table (contains the flashcard data)
notes = pd.read_sql('SELECT * FROM notes;', conn)
print(f'Number of notes: {len(notes)}')
print('\nColumns in notes table:')
print(notes.columns.tolist())

In [ ]:
# Show the first few rows to see what fields are available
print('First 5 notes - all columns:')
print(notes.head())

# The 'flds' column contains the actual flashcard fields separated by \x1f
print('\nFirst note fields (flds column):')
print(notes['flds'].iloc[0].split('\x1f'))

In [ ]:
# Get the model (note type) to understand field names
col = pd.read_sql('SELECT * FROM col;', conn)
# Parse the models JSON
models = json.loads(col['models'].iloc[0])

# Get field names from the first model
for model_id, model in models.items():
    print(f"Model: {model['name']}")
    print(f"Fields: {[f['name'] for f in model['flds']]}")
    break

In [ ]:
# Extract fields into separate columns
field_names = [f['name'] for f in list(models.values())[0]['flds']]

# Split the flds column into separate fields
fields_df = notes['flds'].str.split('\x1f', expand=True)
fields_df.columns = field_names[:len(fields_df.columns)]

# Combine with other useful columns
flashcards = pd.concat([
    notes[['id', 'mod', 'tags']],
    fields_df
], axis=1)

print(f'Total flashcards: {len(flashcards)}')
print('\nFirst 5 flashcards:')
print(flashcards.head())

In [ ]:
# Show sample of the data with all fields
print('Sample flashcard data:')
print(flashcards.iloc[0])

# Show first 10 rows of just the fields
print('\nFirst 10 flashcards (fields only):')
print(flashcards[field_names].head(10))